In [ ]:
import ast
import threading
from flask import Flask, request, render_template_string
import pandas as pd
import numpy as np
from neo4j import GraphDatabase
from neo4j.exceptions import ServiceUnavailable
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

#########################################
# 1. Neo4j Connection and Interaction Agent
#########################################
class Neo4jAgent:
    def __init__(self, uri, user, password):
        self.uri = uri
        self.user = user
        self.password = password
        self.driver = None
        self.connect()
        
    def connect(self):
        try:
            self.driver = GraphDatabase.driver(self.uri, auth=(self.user, self.password))
            with self.driver.session() as session:
                session.run("RETURN 1")
            print("Connected to Neo4j successfully.")
        except ServiceUnavailable as e:
            print("Neo4j connection error:", e)
            exit(1)
    
    def execute_write(self, func, *args, **kwargs):
        with self.driver.session() as session:
            session.execute_write(func, *args, **kwargs)
    
    def run_query(self, query, parameters=None):
        with self.driver.session() as session:
            result = session.run(query, parameters)
            return list(result)
    
    def get_node_count(self):
        result = self.run_query("MATCH (n) RETURN COUNT(n) AS count")
        return result[0]["count"] if result else 0

#########################################
# 2. Graph Builder Agent (Data Ingestion & Relationship Creation)
#########################################
class GraphBuilderAgent:
    def __init__(self, neo4j_agent, csv_paths):
        self.neo4j = neo4j_agent
        self.csv_paths = csv_paths  # a dict mapping data types to CSV file paths
    
    # Node creation functions (each as a static method)
    @staticmethod
    def create_city_node(tx, props):
        query = """
        MERGE (c:City {city_id: $city_id})
        SET c += $props
        """
        tx.run(query, city_id=props["city_id"], props=props)
    
    @staticmethod
    def create_flight_node(tx, props):
        query = """
        MERGE (f:Flight {flight_id: $flight_id})
        SET f += $props
        """
        tx.run(query, flight_id=props["flight_id"], props=props)
    
    @staticmethod
    def create_hotel_node(tx, props):
        query = """
        MERGE (h:Hotel {hotel_id: $hotel_id})
        SET h += $props
        """
        tx.run(query, hotel_id=props["hotel_id"], props=props)
    
    @staticmethod
    def create_restaurant_node(tx, props):
        query = """
        MERGE (r:Restaurant {restaurant_id: $restaurant_id})
        SET r += $props
        """
        tx.run(query, restaurant_id=props["restaurant_id"], props=props)
    
    @staticmethod
    def create_preference_node(tx, props):
        query = """
        MERGE (p:Preference {preference_id: $preference_id})
        SET p += $props
        """
        tx.run(query, preference_id=props["preference_id"], props=props)
    
    @staticmethod
    def create_user_node(tx, props):
        user_id = props.get("User_ID") or props.get("user_id")
        if not user_id:
            raise KeyError("User_ID not found in props for user node")
        query = """
        MERGE (u:User {User_ID: $user_id})
        SET u += $props
        """
        tx.run(query, user_id=user_id, props=props)
    
    @staticmethod
    def create_passport_node(tx, props):
        query = """
        MERGE (pp:Passport {passport_id: $passport_id})
        SET pp += $props
        """
        tx.run(query, passport_id=props["passport_id"], props=props)
    
    @staticmethod
    def create_history_node(tx, props):
        query = """
        MERGE (h:History {history_id: $history_id})
        SET h += $props
        """
        tx.run(query, history_id=props["history_id"], props=props)
    
    @staticmethod
    def create_relationship(tx, label_from, key_from, value_from, rel_type, label_to, key_to, value_to):
        query = f"""
        MATCH (a:{label_from} {{{key_from}: $value_from}})
        MATCH (b:{label_to} {{{key_to}: $value_to}})
        MERGE (a)-[r:{rel_type}]->(b)
        """
        tx.run(query, value_from=value_from, value_to=value_to)
    
    def build_graph(self):
        # Load CSV files
        cities_df = pd.read_csv(self.csv_paths["cities"])
        flights_df = pd.read_csv(self.csv_paths["flights"])
        hotels_df = pd.read_csv(self.csv_paths["hotels"])
        restaurants_df = pd.read_csv(self.csv_paths["restaurants"])
        preferences_df = pd.read_csv(self.csv_paths["preferences"])
        users_df = pd.read_csv(self.csv_paths["users"])
        passports_df = pd.read_csv(self.csv_paths["passports"])
        histories_df = pd.read_csv(self.csv_paths["histories"])
        
        # Create nodes
        for _, row in cities_df.iterrows():
            self.neo4j.execute_write(self.create_city_node, row.to_dict())
        for _, row in flights_df.iterrows():
            self.neo4j.execute_write(self.create_flight_node, row.to_dict())
        for _, row in hotels_df.iterrows():
            self.neo4j.execute_write(self.create_hotel_node, row.to_dict())
        for _, row in restaurants_df.iterrows():
            self.neo4j.execute_write(self.create_restaurant_node, row.to_dict())
        for _, row in preferences_df.iterrows():
            self.neo4j.execute_write(self.create_preference_node, row.to_dict())
        for _, row in users_df.iterrows():
            self.neo4j.execute_write(self.create_user_node, row.to_dict())
        for _, row in passports_df.iterrows():
            self.neo4j.execute_write(self.create_passport_node, row.to_dict())
        for _, row in histories_df.iterrows():
            self.neo4j.execute_write(self.create_history_node, row.to_dict())
        
        # Create relationships based on CSV data
        # Example: History -> Hotel ("STAYED_AT")
        for _, row in histories_df.iterrows():
            hist_id = row.get("history_id")
            hotels_str = row.get("hotels")
            if pd.notnull(hist_id) and isinstance(hotels_str, str) and hotels_str.strip():
                try:
                    hotel_ids = ast.literal_eval(hotels_str)
                    for h_id in hotel_ids:
                        self.neo4j.execute_write(
                            self.create_relationship,
                            "History", "history_id", hist_id,
                            "STAYED_AT",
                            "Hotel", "hotel_id", h_id
                        )
                except Exception as e:
                    print("Error parsing hotels array:", e)
        
        # DINED_AT (History -> Restaurant)
        for _, row in histories_df.iterrows():
            hist_id = row.get("history_id")
            rest_str = row.get("restaurants")
            if pd.notnull(hist_id) and isinstance(rest_str, str) and rest_str.strip():
                try:
                    rest_ids = ast.literal_eval(rest_str)
                    for r_id in rest_ids:
                        self.neo4j.execute_write(
                            self.create_relationship,
                            "History", "history_id", hist_id,
                            "DINED_AT",
                            "Restaurant", "restaurant_id", r_id
                        )
                except Exception as e:
                    print("Error parsing restaurants array:", e)
        
        # HAS_HOTEL_PREFERENCE (Preference -> Hotel)
        for _, row in preferences_df.iterrows():
            pref_id = row.get("preference_id")
            top_hotels_str = row.get("top_hotels")
            if pd.notnull(pref_id) and isinstance(top_hotels_str, str) and top_hotels_str.strip():
                try:
                    hotel_ids = ast.literal_eval(top_hotels_str)
                    for h_id in hotel_ids:
                        self.neo4j.execute_write(
                            self.create_relationship,
                            "Preference", "preference_id", pref_id,
                            "HAS_HOTEL_PREFERENCE",
                            "Hotel", "hotel_id", h_id
                        )
                except Exception as e:
                    print("Error parsing top_hotels array:", e)
        
        # HAS_RESTAURANT_PREFERENCE (Preference -> Restaurant)
        for _, row in preferences_df.iterrows():
            pref_id = row.get("preference_id")
            top_rest_str = row.get("top_restaurants")
            if pd.notnull(pref_id) and isinstance(top_rest_str, str) and top_rest_str.strip():
                try:
                    rest_ids = ast.literal_eval(top_rest_str)
                    for r_id in rest_ids:
                        self.neo4j.execute_write(
                            self.create_relationship,
                            "Preference", "preference_id", pref_id,
                            "HAS_RESTAURANT_PREFERENCE",
                            "Restaurant", "restaurant_id", r_id
                        )
                except Exception as e:
                    print("Error parsing top_restaurants array:", e)
        
        # HAS_CITY_PREFERENCE (Preference -> City)
        for _, row in preferences_df.iterrows():
            pref_id = row.get("preference_id")
            top_cities_str = row.get("top_cities")
            if pd.notnull(pref_id) and isinstance(top_cities_str, str) and top_cities_str.strip():
                try:
                    city_names = ast.literal_eval(top_cities_str)
                    for cname in city_names:
                        self.neo4j.execute_write(
                            self.create_relationship,
                            "Preference", "preference_id", pref_id,
                            "HAS_CITY_PREFERENCE",
                            "City", "City", cname
                        )
                except Exception as e:
                    print("Error parsing top_cities array:", e)
        
        # IS_READY_TO_APPLY_VISA (Preference -> Passport)
        prefs = preferences_df.to_dict("records")
        pports = passports_df.to_dict("records")
        for pref_row in prefs:
            pref_id = pref_row["preference_id"]
            v_pref = pref_row.get("visa_preference")
            if pd.notnull(pref_id) and pd.notnull(v_pref):
                for pport_row in pports:
                    pport_id = pport_row["passport_id"]
                    req = pport_row.get("Requirement")
                    if pd.notnull(pport_id) and pd.notnull(req):
                        if v_pref.strip() == req.strip():
                            self.neo4j.execute_write(
                                self.create_relationship,
                                "Preference", "preference_id", pref_id,
                                "IS_READY_TO_APPLY_VISA",
                                "Passport", "passport_id", pport_id
                            )
        
        # REQUIRED_VISA_LIKE (Passport -> History)
        for pport_row in pports:
            pport_id = pport_row["passport_id"]
            origin = pport_row.get("Origin")
            if pd.notnull(pport_id) and pd.notnull(origin):
                for _, hist_row in histories_df.iterrows():
                    hist_id = hist_row["history_id"]
                    issued_p = hist_row.get("issued_passport")
                    if pd.notnull(hist_id) and pd.notnull(issued_p):
                        if origin.strip() == issued_p.strip():
                            self.neo4j.execute_write(
                                self.create_relationship,
                                "Passport", "passport_id", pport_id,
                                "REQUIRED_VISA_LIKE",
                                "History", "history_id", hist_id
                            )
        print("Graph build complete! All relationships merged.")

#########################################
# 3. Representation Agent (Converts nodes to human‐readable strings)
#########################################
class RepresentationAgent:
    @staticmethod
    def build_representation(props, fields):
        parts = []
        for field, label in fields.items():
            value = props.get(field)
            if value is not None and str(value).strip() != "":
                parts.append(f"{label}: {value}")
        return "; ".join(parts)
    
    @classmethod
    def represent_city(cls, node):
        fields = {
            "City": "City", "Country": "Country",
            "Remote connection: Average WiFi speed (Mbps per second)": "WiFi Speed",
            "Co-working spaces: Number of co-working spaces": "Co-working Spaces",
            "Accommodation: Average price of 1 bedroom apartment per month": "Apartment Price",
            "Food: Average cost of a meal at a local, mid-level restaurant": "Meal Cost",
            "Tourist attractions: Number of Things to do on Tripadvisor": "Attractions"
        }
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def represent_flight(cls, node):
        fields = {
            "Airline": "Airline", "Total Fare (EUR)": "Price",
            "Departure Airport Code": "From", "Arrival Airport Code": "To",
            "Duration (hrs)": "Duration", "Class": "Class"
        }
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def represent_hotel(cls, node):
        fields = {
            "name": "Name", "price": "Price",
            "number_reviews": "Reviews", "City": "City",
            "rating": "Rating", "address": "Address"
        }
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def represent_restaurant(cls, node):
        fields = {
            "Restaurant Name": "Name", "Cuisines": "Cuisines",
            "Average Cost for two": "Price for Two", "City": "City",
            "Aggregate rating": "Rating", "Address": "Address"
        }
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def represent_preference(cls, node):
        fields = {"preference_id": "Preference ID", "visa_preference": "Visa Preference"}
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def represent_user(cls, node):
        fields = {"User_ID": "User ID", "Username": "Username"}
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def represent_passport(cls, node):
        fields = {"passport_id": "Passport ID", "Origin": "Origin", "Requirement": "Requirement"}
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def represent_history(cls, node):
        fields = {"history_id": "History ID", "city": "City", "country": "Country"}
        return cls.build_representation(node._properties, fields)
    
    @classmethod
    def get_all_representations(cls, neo4j_agent):
        all_nodes = []
        label_funcs = [
            ("City", cls.represent_city),
            ("Flight", cls.represent_flight),
            ("Hotel", cls.represent_hotel),
            ("Restaurant", cls.represent_restaurant),
            ("Preference", cls.represent_preference),
            ("User", cls.represent_user),
            ("Passport", cls.represent_passport),
            ("History", cls.represent_history)
        ]
        for label, func in label_funcs:
            result = neo4j_agent.run_query(f"MATCH (n:{label}) RETURN n")
            for record in result:
                node = record["n"]
                rep = func(node)
                if rep:
                    all_nodes.append(rep)
        return list(set(all_nodes))

#########################################
# 4. Embedding Agent (For document retrieval)
#########################################
class EmbeddingAgent:
    def __init__(self):
        print("Computing embeddings...")
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = None
        self.documents = []
    
    def update_documents(self, documents):
        self.documents = documents
        self.doc_embeddings = self.embedder.encode(documents, convert_to_tensor=True)
    
    def retrieve_documents(self, query, top_k=8):
        query_embedding = self.embedder.encode([query], convert_to_tensor=True)
        cos_scores = cosine_similarity(query_embedding.cpu().numpy(), self.doc_embeddings.cpu().numpy())[0]
        sorted_indices = np.argsort(cos_scores)[::-1]
        retrieved_docs = [self.documents[i] for i in sorted_indices[:top_k]]
        return retrieved_docs

#########################################
# 5. Query Agent (Detects and refines queries)
#########################################
class QueryAgent:
    def __init__(self):
        self.generator = pipeline(
            "text-generation",
            model="gpt2",
            do_sample=True,
            temperature=0.7,
            max_new_tokens=200,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2
        )
    
    def detect_query_type(self, query):
        query_lower = query.lower()
        if any(word in query_lower for word in ["hotel", "stay", "accommodation", "lodging"]):
            return "hotel"
        elif any(word in query_lower for word in ["restaurant", "eat", "dine", "food", "cuisine"]):
            return "restaurant"
        elif any(word in query_lower for word in ["flight", "fly", "airline", "ticket"]):
            return "flight"
        elif any(word in query_lower for word in ["city", "destination", "place", "visit", "location"]):
            return "city"
        elif any(word in query_lower for word in ["trip", "itinerary", "plan", "vacation", "holiday"]):
            return "complete_trip"
        elif any(word in query_lower for word in ["clear", "reset", "delete", "erase"]):
            return "clear_history"
        else:
            return "general"
    
    def refine_query(self, raw_query, query_type):
        if query_type == "clear_history":
            return raw_query
        prompt = f"""
        Refine this travel query to be more specific for a {query_type} search:
        Original Query: {raw_query}
        Refined Query:"""
        result = self.generator(prompt, num_return_sequences=1)
        refined = result[0]["generated_text"].replace(prompt, "").strip().split("\n")[0].strip()
        return refined

#########################################
# 6. Response Agent (Queries Neo4j, retrieves docs, and generates responses)
#########################################
class ResponseAgent:
    def __init__(self, neo4j_agent, embedding_agent, query_agent):
        self.neo4j = neo4j_agent
        self.embedding_agent = embedding_agent
        self.query_agent = query_agent
    
    def query_neo4j(self, query_type, filters=None):
        if filters is None:
            filters = {}
        results = []
        if query_type == "hotel":
            cypher = "MATCH (h:Hotel) WHERE 1=1"
            if 'city' in filters:
                cypher += f" AND h.City = '{filters['city']}'"
            if 'max_price' in filters:
                cypher += f" AND toFloat(h.price) <= {filters['max_price']}"
            cypher += " RETURN h"
            for record in self.neo4j.run_query(cypher):
                results.append(record["h"])
        elif query_type == "restaurant":
            cypher = "MATCH (r:Restaurant) WHERE 1=1"
            if 'city' in filters:
                cypher += f" AND r.City = '{filters['city']}'"
            if 'cuisine' in filters:
                cypher += f" AND toLower(r.Cuisines) CONTAINS toLower('{filters['cuisine']}')"
            cypher += " RETURN r"
            for record in self.neo4j.run_query(cypher):
                results.append(record["r"])
        elif query_type == "flight":
            cypher = "MATCH (f:Flight) WHERE 1=1"
            if 'destination' in filters:
                cypher += f" AND toLower(f.`Arrival Airport Code`) = toLower('{filters['destination']}')"
            if 'max_price' in filters:
                cypher += f" AND toFloat(f.`Total Fare (EUR)`) <= {filters['max_price']}"
            cypher += " RETURN f"
            for record in self.neo4j.run_query(cypher):
                results.append(record["f"])
        elif query_type == "city":
            cypher = "MATCH (c:City) WHERE 1=1"
            if 'country' in filters:
                cypher += f" AND toLower(c.Country) = toLower('{filters['country']}')"
            if 'wifi_speed' in filters:
                cypher += f" AND toFloat(c.`Remote connection: Average WiFi speed (Mbps per second)`) >= {filters['wifi_speed']}"
            cypher += " RETURN c"
            for record in self.neo4j.run_query(cypher):
                results.append(record["c"])
        return results
    
    def generate_response(self, query):
        refined_query = self.query_agent.refine_query(query, self.query_agent.detect_query_type(query))
        query_type = self.query_agent.detect_query_type(query)
        print(f"Detected query type: {query_type}, Refined: {refined_query}")
        
        if query_type == "clear_history":
            return "I've cleared our conversation history. How can I help you with your travel plans?", []
        
        filters = {}
        if query_type == "hotel":
            result = self.neo4j.run_query("MATCH (c:City) RETURN c.City as city")
            cities = [record["city"] for record in result]
            for city in cities:
                if city is not None and city.lower() in refined_query.lower():
                    filters['city'] = city
                    break
            if "under" in refined_query.lower() and "$" in refined_query.lower():
                try:
                    max_price = float(refined_query.split("$")[1].split()[0])
                    filters['max_price'] = max_price
                except:
                    pass
        elif query_type == "restaurant":
            result = self.neo4j.run_query("MATCH (c:City) RETURN c.City as city")
            cities = [record["city"] for record in result]
            for city in cities:
                if city is not None and city.lower() in refined_query.lower():
                    filters['city'] = city
                    break
            cuisine_words = ["italian", "chinese", "french", "japanese", "mexican", "indian", "thai"]
            for word in cuisine_words:
                if word in refined_query.lower():
                    filters['cuisine'] = word
                    break
        elif query_type == "flight":
            result = self.neo4j.run_query("MATCH (c:City) RETURN c.City as city")
            cities = [record["city"] for record in result]
            for city in cities:
                if city is not None and city.lower() in refined_query.lower():
                    filters['destination'] = city
                    break
            if "under" in refined_query.lower() and "$" in refined_query.lower():
                try:
                    max_price = float(refined_query.split("$")[1].split()[0])
                    filters['max_price'] = max_price
                except:
                    pass
        elif query_type == "city":
            result = self.neo4j.run_query("MATCH (c:City) RETURN c.Country as country")
            countries = [record["country"] for record in result]
            for country in countries:
                if country is not None and country.lower() in refined_query.lower():
                    filters['country'] = country
                    break
            if "wifi" in refined_query.lower() or "internet" in refined_query.lower():
                filters['wifi_speed'] = 50
        
        neo4j_results = self.query_neo4j(query_type, filters)
        retrieved_docs = self.embedding_agent.retrieve_documents(refined_query, top_k=10)
        
        if not neo4j_results and not retrieved_docs:
            return "I couldn't find enough information about that. Could you be more specific?", []
        
        if query_type == "hotel":
            if not neo4j_results:
                return "I couldn't find any hotels matching your criteria in our database. Please try a different search.", []
            neo4j_results.sort(key=lambda x: float(x._properties.get("price", 99999)))
            target_city = filters.get('city', 'our database')
            max_price = filters.get('max_price', 'any price')
            response = f"Here are the best hotel options in {target_city} under ${max_price if max_price != 'any price' else 'any price'}:\n\n"
            for hotel in neo4j_results[:5]:
                props = hotel._properties
                response += f"🏨 {props.get('name', 'N/A')}\n"
                response += f"   - Price: ${props.get('price', 'N/A')}\n"
                response += f"   - Reviews: {props.get('number_reviews', 'N/A')}\n"
                response += f"   - Rating: {props.get('rating', 'N/A')}\n"
                response += f"   - Address: {props.get('address', 'N/A')}\n\n"
            response += "Would you like:\n"
            response += "1. More details about any of these hotels\n"
            response += "2. Cheaper options in a different area\n"
            response += "3. Higher-end options with better amenities\n"
            response += "4. Something else?"
            return response, [RepresentationAgent.represent_hotel(h) for h in neo4j_results[:5]]
        
        elif query_type == "restaurant":
            if not neo4j_results:
                cuisine = filters.get('cuisine', '')
                city = filters.get('city', 'our database')
                return f"I couldn't find any {cuisine + ' ' if cuisine else ''}restaurants matching your criteria in {city}. Please try a different search.", []
            neo4j_results.sort(key=lambda x: float(x._properties.get("Average Cost for two", 0)))
            cuisine = filters.get('cuisine', '')
            city = filters.get('city', 'various cities')
            response = f"Here are some excellent {cuisine if cuisine else ''} restaurant options in {city}:\n\n"
            for restaurant in neo4j_results[:5]:
                props = restaurant._properties
                response += f"🍽️ {props.get('Restaurant Name', 'N/A')}\n"
                response += f"   - Cuisine: {props.get('Cuisines', 'N/A')}\n"
                response += f"   - Avg. cost for two: ${props.get('Average Cost for two', 'N/A')}\n"
                response += f"   - Rating: {props.get('Aggregate rating', 'N/A')}\n"
                response += f"   - Address: {props.get('Address', 'N/A')}\n\n"
            response += "Would you like to:\n"
            response += "1. Filter by a specific price range\n"
            response += "2. See options in a different area\n"
            response += "3. Get recommendations for a different cuisine\n"
            response += "4. More details about any of these"
            return response, [RepresentationAgent.represent_restaurant(r) for r in neo4j_results[:5]]
        
        elif query_type == "flight":
            if not neo4j_results:
                return "I couldn't find any flights matching your criteria. Please try a different search.", []
            neo4j_results.sort(key=lambda x: float(x._properties.get("Total Fare (EUR)", 99999)))
            destination = filters.get('destination', 'various destinations')
            response = f"Here are the best flight options to {destination}:\n\n"
            for flight in neo4j_results[:5]:
                props = flight._properties
                response += f"✈️ {props.get('Airline', 'N/A')}\n"
                response += f"   - From: {props.get('Departure Airport Code', 'N/A')}\n"
                response += f"   - To: {props.get('Arrival Airport Code', 'N/A')}\n"
                response += f"   - Price: ${props.get('Total Fare (EUR)', 'N/A')}\n"
                response += f"   - Duration: {props.get('Duration (hrs)', 'N/A')} hours\n"
                response += f"   - Class: {props.get('Class', 'N/A')}\n\n"
            response += "Would you like to:\n"
            response += "1. See flights from a specific location\n"
            response += "2. Filter by airline or flight duration\n"
            response += "3. See business class options\n"
            response += "4. Get recommendations for a different destination"
            return response, [RepresentationAgent.represent_flight(f) for f in neo4j_results[:5]]
        
        elif query_type == "city":
            if not neo4j_results:
                return "I couldn't find any cities matching your criteria. Please try a different search.", []
            response = "Here are some great travel destinations:\n\n"
            for city in neo4j_results[:5]:
                props = city._properties
                response += f"🌆 {props.get('City', 'N/A')}, {props.get('Country', 'N/A')}\n"
                response += f"   - Avg. apartment price: ${props.get('Accommodation: Average price of 1 bedroom apartment per month', 'N/A')}/month\n"
                response += f"   - Avg. meal cost: ${props.get('Food: Average cost of a meal at a local, mid-level restaurant', 'N/A')}\n"
                response += f"   - WiFi speed: {props.get('Remote connection: Average WiFi speed (Mbps per second)', 'N/A')} Mbps\n"
                response += f"   - Attractions: {props.get('Tourist attractions: Number of Things to do on Tripadvisor', 'N/A')} things to do\n\n"
            response += "Would you like more details about:\n"
            response += "1. Digital nomad-friendly cities\n"
            response += "2. Budget travel destinations\n"
            response += "3. Luxury travel options\n"
            response += "4. A specific city"
            return response, [RepresentationAgent.represent_city(c) for c in neo4j_results[:5]]
        
        elif query_type == "complete_trip":
            destination = None
            duration = 5  # default duration
            duration_words = ["day", "week", "month"]
            for word in duration_words:
                if word in refined_query.lower():
                    try:
                        duration = int(refined_query.lower().split(word)[0].split()[-1])
                        if word == "week":
                            duration *= 7
                        elif word == "month":
                            duration *= 30
                    except:
                        pass
            result = self.neo4j.run_query("MATCH (c:City) RETURN c")
            for record in result:
                city = record["c"]
                city_name = city._properties.get("City") or ""
                if city_name and city_name.lower() in refined_query.lower():
                    destination = city
                    break
            if not destination:
                return "Please specify a destination city for your trip plan (e.g., 'Plan a 5-day trip to Paris').", []
            hotels_result = self.neo4j.run_query("MATCH (h:Hotel) WHERE h.City = $city RETURN h", {"city": destination._properties.get("City")})
            restaurants_result = self.neo4j.run_query("MATCH (r:Restaurant) WHERE r.City = $city RETURN r", {"city": destination._properties.get("City")})
            flights_result = self.neo4j.run_query("MATCH (f:Flight) WHERE toLower(f.`Arrival Airport Code`) = toLower($city) RETURN f", {"city": destination._properties.get("City")})
            hotels = [record["h"] for record in hotels_result]
            restaurants = [record["r"] for record in restaurants_result]
            flights = [record["f"] for record in flights_result]
            props = destination._properties
            response = f"Here's a suggested {duration}-day itinerary for {props.get('City', 'N/A')}, {props.get('Country', 'N/A')}:\n\n"
            response += "📅 Day 1: Arrival & First Impressions\n"
            if flights:
                flight_props = flights[0]._properties
                response += f"✈️ Flight: {flight_props.get('Airline', 'N/A')} from {flight_props.get('Departure Airport Code', 'N/A')} for ${flight_props.get('Total Fare (EUR)', 'N/A')} ({flight_props.get('Duration (hrs)', 'N/A')} hrs)\n"
            if hotels:
                hotel_props = hotels[len(hotels)//2]._properties
                response += f"🏨 Hotel: {hotel_props.get('name', 'N/A')} (${hotel_props.get('price', 'N/A')}/night, {hotel_props.get('rating', 'N/A')}★)\n"
                response += f"   - Address: {hotel_props.get('address', 'N/A')}\n"
            response += "   - After checking in, take a walk around the neighborhood to get oriented\n"
            if restaurants:
                restaurant_props = restaurants[0]._properties
                response += f"🍽️ Dinner: {restaurant_props.get('Restaurant Name', 'N/A')} ({restaurant_props.get('Cuisines', 'N/A')}, ${restaurant_props.get('Average Cost for two', 'N/A')} for two)\n"
                response += f"   - Rating: {restaurant_props.get('Aggregate rating', 'N/A')}★\n\n"
            response += f"📅 Day 2: Explore {props.get('City', 'N/A')}\n"
            response += "   - Morning: Visit top historical attractions\n"
            response += "   - Afternoon: Guided tour or local market exploration\n"
            response += "   - Evening: Enjoy local entertainment or nightlife\n"
            if len(restaurants) > 1:
                restaurant_props = restaurants[1]._properties
                response += f"🍽️ Dinner: {restaurant_props.get('Restaurant Name', 'N/A')} ({restaurant_props.get('Cuisines', 'N/A')}, ${restaurant_props.get('Average Cost for two', 'N/A')} for two)\n\n"
            response += "📅 Day 3: Cultural Immersion\n"
            response += "   - Morning: Visit museums or cultural centers\n"
            response += "   - Afternoon: Cooking class or craft workshop\n"
            response += "   - Evening: Traditional performance\n\n"
            response += "📅 Day 4: Day Trip\n"
            response += "   - Full-day excursion to nearby attractions\n\n"
            response += "📅 Day 5: Relaxation & Departure\n"
            response += "   - Morning: Last-minute shopping or revisit favorite spots\n"
            response += "   - Afternoon: Check out from hotel\n"
            if flights:
                flight_props = flights[0]._properties
                response += f"✈️ Flight: {flight_props.get('Airline', 'N/A')} to {flight_props.get('Departure Airport Code', 'N/A')}\n\n"
            total_cost = 0
            if flights:
                total_cost += float(flights[0]._properties.get("Total Fare (EUR)", 0)) * 2
            if hotels:
                total_cost += float(hotels[len(hotels)//2]._properties.get("price", 0)) * duration
            if restaurants:
                total_cost += float(restaurants[0]._properties.get("Average Cost for two", 0)) * duration / 2
            total_cost += 50 * duration
            response += f"💰 Estimated total cost for this trip: ${total_cost:.2f} (for one person)\n\n"
            response += "Would you like me to:\n"
            response += "1. Adjust this itinerary (budget changes)\n"
            response += "2. Focus on specific interests (culture, food, adventure)\n"
            response += "3. Provide more detailed daily activities\n"
            response += "4. Book any of these options"
            retrieved = []
            if flights: retrieved.append(RepresentationAgent.represent_flight(flights[0]))
            if hotels: retrieved.append(RepresentationAgent.represent_hotel(hotels[len(hotels)//2]))
            if restaurants:
                retrieved.extend([RepresentationAgent.represent_restaurant(r) for r in restaurants[:2]])
            retrieved.append(RepresentationAgent.represent_city(destination))
            return response, retrieved
        
        else:
            prompt = f"""You are a knowledgeable travel assistant. Provide a helpful, detailed response to this travel question:

Question: {query}

Available information:
{retrieved_docs}

Response:"""
            result = self.query_agent.generator(prompt, num_return_sequences=1)
            response = result[0]["generated_text"].replace(prompt, "").strip()
            return response, retrieved_docs

#########################################
# 7. Travel Assistant Agent (Orchestrates everything)
#########################################
class TravelAssistantAgent:
    def __init__(self, neo4j_uri, neo4j_user, neo4j_password, csv_paths):
        self.neo4j_agent = Neo4jAgent(neo4j_uri, neo4j_user, neo4j_password)
        self.graph_builder = GraphBuilderAgent(self.neo4j_agent, csv_paths)
        self.representation_agent = RepresentationAgent
        self.embedding_agent = EmbeddingAgent()
        self.query_agent = QueryAgent()
        self.response_agent = ResponseAgent(self.neo4j_agent, self.embedding_agent, self.query_agent)
        self.conversation_history = []
        
        # Build the graph (run once; comment out after first run if desired)
        self.graph_builder.build_graph()
        # Update embedding documents from node representations
        docs = self.representation_agent.get_all_representations(self.neo4j_agent)
        self.embedding_agent.update_documents(docs)
    
    def handle_query(self, query):
        query_type = self.query_agent.detect_query_type(query)
        self.conversation_history.append({
            "sender": "User",
            "text": query,
            "query_type": query_type.replace("_", " ").title()
        })
        answer, retrieved_data = self.response_agent.generate_response(query)
        self.conversation_history.append({
            "sender": "Assistant",
            "text": answer
        })
        return answer, retrieved_data

#########################################
# 8. Flask Web Interface (Enhanced version with custom HTML)
#########################################
app = Flask(__name__)

HTML_TEMPLATE = """
<!DOCTYPE html>
<html>
<head>
    <title>Neo4j Travel Assistant</title>
    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
            background-color: #f5f7fa;
            color: #333;
        }
        .header {
            background-color: #4285f4;
            color: white;
            padding: 20px;
            border-radius: 8px;
            margin-bottom: 20px;
            text-align: center;
        }
        .filter-section {
            background-color: white;
            padding: 15px;
            border-radius: 8px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }
        .filter-buttons {
            display: flex;
            gap: 10px;
            flex-wrap: wrap;
            margin-top: 10px;
        }
        .filter-button {
            padding: 8px 15px;
            background-color: #e0e0e0;
            border: none;
            border-radius: 20px;
            cursor: pointer;
            transition: background-color 0.3s;
        }
        .filter-button:hover {
            background-color: #d0d0d0;
        }
        .filter-button.active {
            background-color: #4285f4;
            color: white;
        }
        .chat-container {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            height: 500px;
            overflow-y: auto;
            white-space: pre-wrap;
        }
        .message {
            margin-bottom: 15px;
            padding: 10px 15px;
            border-radius: 18px;
            max-width: 80%;
            word-wrap: break-word;
        }
        .user-message {
            background-color: #e3f2fd;
            margin-left: auto;
            border-bottom-right-radius: 4px;
        }
        .bot-message {
            background-color: #f1f1f1;
            margin-right: auto;
            border-bottom-left-radius: 4px;
            white-space: pre-wrap;
        }
        .data-section {
            background-color: white;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            max-height: 300px;
            overflow-y: auto;
        }
        .data-item {
            padding: 10px;
            border-bottom: 1px solid #eee;
            font-family: monospace;
        }
        .input-section {
            display: flex;
            gap: 10px;
        }
        #user-input {
            flex-grow: 1;
            padding: 12px;
            border: 1px solid #ddd;
            border-radius: 8px;
            font-size: 16px;
        }
        #submit-button {
            padding: 12px 20px;
            background-color: #4285f4;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 16px;
        }
        #submit-button:hover {
            background-color: #3367d6;
        }
        .query-type-indicator {
            font-size: 14px;
            color: #666;
            margin-top: 5px;
            font-style: italic;
        }
        .clear-button {
            padding: 8px 15px;
            background-color: #f44336;
            color: white;
            border: none;
            border-radius: 8px;
            cursor: pointer;
            font-size: 14px;
            margin-top: 10px;
        }
        .clear-button:hover {
            background-color: #d32f2f;
        }
        .neo4j-info {
            background-color: #008cc1;
            color: white;
            padding: 10px;
            border-radius: 5px;
            margin-top: 10px;
            font-size: 14px;
        }
    </style>
</head>
<body>
    <div class="header">
        <h1>Neo4j Travel Assistant</h1>
        <p>Get personalized travel recommendations powered by Neo4j graph database</p>
        <div class="neo4j-info">
            Connected to Neo4j at {{ NEO4J_URI }} with {{ node_count }} nodes in database
        </div>
    </div>
    
    <div class="filter-section">
        <h3>Not sure what to ask? Try these Neo4j-powered queries:</h3>
        <div class="filter-buttons">
            <button class="filter-button" onclick="setQuery('Best hotels in New York under $200')">Hotels</button>
            <button class="filter-button" onclick="setQuery('Italian restaurants in London')">Restaurants</button>
            <button class="filter-button" onclick="setQuery('Cheapest flights to Dubai next month')">Flights</button>
            <button class="filter-button" onclick="setQuery('Best digital nomad cities with good WiFi')">Destinations</button>
            <button class="filter-button" onclick="setQuery('Plan a complete 5-day trip to Istanbul')">Complete Trip</button>
        </div>
        <button class="clear-button" onclick="setQuery('clear history')">Clear Conversation</button>
    </div>
    
    <div class="chat-container" id="chat-container">
        {% for msg in history %}
            <div class="message {% if msg.sender == 'User' %}user-message{% else %}bot-message{% endif %}">
                <strong>{{ msg.sender }}:</strong> {{ msg.text }}
                {% if msg.query_type %}
                <div class="query-type-indicator">Detected as: {{ msg.query_type }}</div>
                {% endif %}
            </div>
        {% endfor %}
    </div>
    
    <div class="data-section">
        <h3>Neo4j Data Details</h3>
        {% if retrieved_data %}
            {% for doc in retrieved_data %}
                <div class="data-item">{{ doc }}</div>
            {% endfor %}
        {% else %}
            <div class="data-item">No data retrieved yet. Ask about hotels, restaurants, flights or destinations.</div>
        {% endif %}
    </div>
    
    <form method="post" class="input-section">
        <input type="text" id="user-input" name="question" placeholder="Ask about hotels, flights, restaurants or destinations..." required>
        <input type="submit" id="submit-button" value="Send">
    </form>
    
    <script>
        function setQuery(query) {
            document.getElementById('user-input').value = query;
            if (query.toLowerCase().includes('clear')) {
                document.forms[0].submit();
            }
            document.getElementById('user-input').focus();
        }
        
        // Auto-scroll chat to bottom
        var chatContainer = document.getElementById("chat-container");
        chatContainer.scrollTop = chatContainer.scrollHeight;
    </script>
</body>
</html>
"""

# Define CSV file paths
CSV_PATHS = {
    "cities": "adjusted_datasets/adjusted_cities.csv",
    "flights": "adjusted_datasets/adjusted_flights.csv",
    "hotels": "adjusted_datasets/adjusted_hotels.csv",
    "restaurants": "adjusted_datasets/adjusted_restaurants.csv",
    "preferences": "adjusted_datasets/preferences.csv",
    "users": "adjusted_datasets/users.csv",
    "passports": "adjusted_datasets/adjusted_passports.csv",
    "histories": "adjusted_datasets/histories.csv"
}

# Initialize the Travel Assistant Agent
assistant = TravelAssistantAgent("bolt://localhost:7687", "neo4j", "argentic", CSV_PATHS)

@app.route("/", methods=["GET", "POST"])
def index():
    retrieved_data = []
    node_count = assistant.neo4j_agent.get_node_count()
    if request.method == "POST":
        question = request.form["question"]
        answer, retrieved_data = assistant.handle_query(question)
    return render_template_string(
        HTML_TEMPLATE,
        history=assistant.conversation_history,
        retrieved_data=retrieved_data,
        NEO4J_URI=assistant.neo4j_agent.uri,
        node_count=node_count
    )


if __name__ == "__main__":
    app.run(port=5001, debug=True, use_reloader=False)



c:\Users\Tristan\anaconda3\envs\travel_app_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Neo4j successfully.
Computing embeddings...


Device set to use cpu


Graph build complete! All relationships merged.
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit
127.0.0.1 - - [27/Mar/2025 22:12:55] "GET / HTTP/1.1" 200 -
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
127.0.0.1 - - [27/Mar/2025 22:13:02] "POST / HTTP/1.1" 200 -


...

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Detected query type: restaurant, Refined: English and French American cuisine (including pizza) at an Indian Restaurant, New York City. All other queries are subject only of the original request which is being returned by your server(s). * The following table provides additional information about our international searches based on availability or price categories; we do not provide any further details as these may change without prior notice from you.* We also publish pricing averages that will apply across all European stores within those regions if applicable because they can vary with different global prices due both geographically & culturally when calculating actual performance costs associated mainly with online shopping services such like Amazon Prime through their website* Prices quoted above have been determined using Google's own estimates according "price inflation" calculations conducted between 2000 - 2015


127.0.0.1 - - [27/Mar/2025 22:13:29] "POST / HTTP/1.1" 200 -
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Detected query type: restaurant, Refined: add my name or address as an option in the menu item and follow it. This may seem like "add" but you can use any other words, such of your choosing on page 6 (section 8). You will need additional information about where each service is located within our database so please refer back when submitting queries! For example: http://www1.example.com/restaurant_id?name=E&address=$(dateTime - lastDayOfWeek) # Search results based solely upon customer visits by visiting www 1 2 3 4 5 6 7 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94


127.0.0.1 - - [27/Mar/2025 22:14:06] "POST / HTTP/1.1" 200 -
